[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-10-capstone-lakehouse.ipynb#scrollTo=aa101001)

---
# Day 10 · Capstone — Analytical Lakehouse Pipeline on Remote Data
**certified-journeys / duckdb-certified** &nbsp;|&nbsp; Capstone Project

> **Goal for today:** Build a complete medallion-architecture lakehouse in DuckDB using publicly available remote Parquet files, SQL transformations with window functions, EXPLAIN ANALYZE profiling, and a Python orchestration script that runs every stage in order.

---
## What you'll build

```
Remote HTTP Parquet (NYC Yellow Taxi)
         │
         ▼
  ┌─────────────┐
  │  RAW layer  │  read_parquet() over HTTP via httpfs
  └──────┬──────┘
         │ type-cast, filter nulls, add derived columns
         ▼
  ┌──────────────────┐
  │  STAGING layer   │  clean, typed, deduplicated
  └────────┬─────────┘
           │ aggregations + window functions
           ▼
  ┌──────────────────┐
  │   GOLD layer     │  analytical fact table → Parquet export
  └──────────────────┘
```

**Dataset:** NYC Yellow Taxi trip records (January 2024) — publicly available Parquet hosted by the NYC Taxi & Limousine Commission.

> **Reference docs used today:**
> - [DuckDB httpfs extension](https://duckdb.org/docs/extensions/httpfs/overview)
> - [DuckDB Performance Best Practices](https://duckdb.org/docs/guides/performance/overview)
> - [DuckDB COPY Statement](https://duckdb.org/docs/sql/statements/copy)
> - [DuckDB Window Functions](https://duckdb.org/docs/sql/window_functions)
> - [DuckDB Query Profiling](https://duckdb.org/docs/dev/profiling)

In [ ]:
%pip install -q duckdb

---
## Step 1 · Setup — load extensions and configure DuckDB

We need the `httpfs` extension to query remote Parquet files over HTTP. DuckDB v0.10+ auto-loads most core extensions, but we'll load it explicitly for clarity.

We also set a reasonable thread count and memory limit for Colab's environment.

In [ ]:
import duckdb
import os
import time
import pathlib

# Use an in-memory database — the pipeline exports to Parquet at the gold layer
con = duckdb.connect()

# Load the httpfs extension for remote file reads
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Colab-friendly thread/memory settings
con.execute("SET threads TO 2")
con.execute("SET memory_limit = '1GB'")

# Confirm extensions are loaded
exts = con.execute("""
    SELECT extension_name, loaded, installed
    FROM duckdb_extensions()
    WHERE extension_name IN ('httpfs', 'parquet')
""").df()
print("Extension status:")
print(exts.to_string(index=False))
print("\nDuckDB version:", duckdb.__version__)

---
## Step 2 · RAW layer — ingest NYC Yellow Taxi data over HTTP

The NYC TLC publishes monthly Yellow Taxi trip data as Parquet files on a public CDN.
We read it directly with `read_parquet()` and `httpfs` — no download needed.

**What the raw table preserves:**
- All original columns, exactly as they arrive
- No filtering, no type coercion — raw is the source of truth
- We materialise it with `CREATE TABLE AS SELECT` so downstream layers don't re-fetch the remote file

> **Medallion principle:** never modify raw data. If the pipeline fails downstream, you can rebuild staging and gold without re-fetching.

In [ ]:
# Public NYC Yellow Taxi data — January 2024
# Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
RAW_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

print(f"Fetching schema from: {RAW_URL}")
print("(This streams the file header only — no full download until CTAS runs)")

# Preview the schema before materialising
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{RAW_URL}') LIMIT 0").df()
print(f"\nColumns ({len(schema)}):")
print(schema[['column_name', 'column_type']].to_string(index=False))

t0 = time.time()

# Materialise raw layer — CTAS streams the Parquet over HTTP
con.execute(f"""
CREATE OR REPLACE TABLE raw_taxi AS
SELECT * FROM read_parquet('{RAW_URL}')
""")

elapsed = time.time() - t0
raw_count = con.execute("SELECT count(*) FROM raw_taxi").fetchone()[0]
print(f"\nRAW layer: {raw_count:,} rows ingested in {elapsed:.1f}s")

---
## Step 3 · STAGING layer — clean, type-cast, and enrich

Staging applies business rules:

| Rule | Why |
|---|---|
| Filter `trip_distance > 0` and `fare_amount > 0` | Remove zero/negative values that indicate data errors |
| Filter `passenger_count BETWEEN 1 AND 8` | Remove nulls and physically impossible passenger counts |
| Cast pickup/dropoff to `TIMESTAMP` | Ensure consistent datetime type for window functions |
| Add `trip_duration_minutes` | Derived column: minutes between pickup and dropoff |
| Add `hour_of_day`, `day_of_week` | Feature columns for the analytical layer |
| Filter `trip_duration_minutes BETWEEN 1 AND 180` | Remove sub-1-minute and 3-hour+ outliers |

All transformations are SQL — no Pandas required.

In [ ]:
t0 = time.time()

con.execute("""
CREATE OR REPLACE TABLE stg_taxi AS
SELECT
    VendorID                                          AS vendor_id,
    tpep_pickup_datetime::TIMESTAMP                   AS pickup_ts,
    tpep_dropoff_datetime::TIMESTAMP                  AS dropoff_ts,
    passenger_count::INTEGER                          AS passenger_count,
    trip_distance::DOUBLE                             AS trip_distance_mi,
    RatecodeID::INTEGER                               AS rate_code,
    PULocationID::INTEGER                             AS pickup_location_id,
    DOLocationID::INTEGER                             AS dropoff_location_id,
    fare_amount::DOUBLE                               AS fare_amount,
    tip_amount::DOUBLE                                AS tip_amount,
    total_amount::DOUBLE                              AS total_amount,
    payment_type::INTEGER                             AS payment_type,
    -- Derived columns
    date_diff('minute', tpep_pickup_datetime::TIMESTAMP,
                        tpep_dropoff_datetime::TIMESTAMP) AS trip_duration_minutes,
    hour(tpep_pickup_datetime::TIMESTAMP)             AS hour_of_day,
    dayofweek(tpep_pickup_datetime::TIMESTAMP)        AS day_of_week,
    CASE payment_type::INTEGER
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        ELSE 'Unknown'
    END                                               AS payment_label
FROM raw_taxi
WHERE trip_distance > 0
  AND fare_amount > 0
  AND total_amount > 0
  AND passenger_count BETWEEN 1 AND 8
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND date_diff('minute', tpep_pickup_datetime::TIMESTAMP,
                          tpep_dropoff_datetime::TIMESTAMP) BETWEEN 1 AND 180
""")

elapsed = time.time() - t0
stg_count = con.execute("SELECT count(*) FROM stg_taxi").fetchone()[0]
raw_count = con.execute("SELECT count(*) FROM raw_taxi").fetchone()[0]
print(f"STAGING layer: {stg_count:,} rows ({stg_count/raw_count*100:.1f}% of raw) in {elapsed:.1f}s")
print()
print("Sample staging rows:")
print(con.execute("""
    SELECT pickup_ts, pickup_location_id, trip_distance_mi,
           trip_duration_minutes, fare_amount, payment_label
    FROM stg_taxi LIMIT 5
""").df().to_string(index=False))

**What just happened?**
- SQL applied all cleaning rules in a single `CREATE TABLE AS SELECT` — no Pandas, no loops
- `date_diff('minute', ...)` computed trip duration inline during the scan
- Derived columns (`hour_of_day`, `day_of_week`, `payment_label`) are computed once at staging time, not repeatedly in downstream queries
- Filtering reduced the dataset by roughly 1–3% — this is normal for TLC data

Let's verify the data quality before proceeding.

In [ ]:
# Data quality summary
quality = con.execute("""
SELECT
    count(*)                                    AS total_trips,
    round(avg(trip_distance_mi), 2)             AS avg_distance_mi,
    round(avg(trip_duration_minutes), 1)        AS avg_duration_min,
    round(avg(fare_amount), 2)                  AS avg_fare,
    round(avg(tip_amount), 2)                   AS avg_tip,
    round(avg(tip_amount / fare_amount * 100) FILTER (WHERE fare_amount > 0), 1) AS avg_tip_pct,
    min(pickup_ts)::DATE                        AS earliest_pickup,
    max(pickup_ts)::DATE                        AS latest_pickup
FROM stg_taxi
""").df()
print("Staging quality summary:")
print(quality.T.to_string(header=False))

print("\nTrips by payment type:")
print(con.execute("""
    SELECT payment_label, count(*) AS trips,
           round(avg(tip_amount), 2) AS avg_tip
    FROM stg_taxi
    GROUP BY payment_label
    ORDER BY trips DESC
""").df().to_string(index=False))

---
## Step 4 · GOLD layer — analytical fact table with window functions

The gold layer answers business questions. We build a `gold_hourly_stats` table that:
1. Aggregates trips by `hour_of_day` and `payment_label`
2. Computes running totals with `SUM(...) OVER (PARTITION BY ...)`
3. Ranks hours by revenue with `RANK() OVER (...)`
4. Computes a 3-hour rolling average fare with `AVG(...) OVER (... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`

This demonstrates all three window function classes: ranking, distribution, and frame-based aggregates.

> **DuckDB tip:** window functions run *after* GROUP BY — wrap your aggregation in a CTE and apply windows in the outer SELECT.

In [ ]:
t0 = time.time()

con.execute("""
CREATE OR REPLACE TABLE gold_hourly_stats AS
WITH hourly_agg AS (
    -- Step 1: aggregate by hour and payment type
    SELECT
        hour_of_day,
        payment_label,
        count(*)                          AS trip_count,
        round(sum(fare_amount), 2)        AS total_fare,
        round(avg(fare_amount), 2)        AS avg_fare,
        round(avg(tip_amount), 2)         AS avg_tip,
        round(avg(trip_distance_mi), 2)   AS avg_distance_mi,
        round(avg(trip_duration_minutes), 1) AS avg_duration_min
    FROM stg_taxi
    GROUP BY hour_of_day, payment_label
),
windowed AS (
    -- Step 2: apply window functions over the aggregated rows
    SELECT
        hour_of_day,
        payment_label,
        trip_count,
        total_fare,
        avg_fare,
        avg_tip,
        avg_distance_mi,
        avg_duration_min,
        -- Running total of trips within each payment type, ordered by hour
        SUM(trip_count) OVER (
            PARTITION BY payment_label
            ORDER BY hour_of_day
        )                                 AS running_trips,
        -- Rank hours by total fare revenue (all payment types combined)
        RANK() OVER (
            ORDER BY total_fare DESC
        )                                 AS fare_rank,
        -- 3-hour rolling average of avg_fare within each payment type
        round(AVG(avg_fare) OVER (
            PARTITION BY payment_label
            ORDER BY hour_of_day
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2)                             AS rolling_3h_avg_fare,
        -- Percent of trips this hour vs. total for this payment type
        round(trip_count * 100.0 /
            SUM(trip_count) OVER (PARTITION BY payment_label), 1) AS pct_of_payment_type
    FROM hourly_agg
)
SELECT * FROM windowed
ORDER BY hour_of_day, payment_label
""")

elapsed = time.time() - t0
gold_count = con.execute("SELECT count(*) FROM gold_hourly_stats").fetchone()[0]
print(f"GOLD layer: {gold_count} rows in {elapsed:.1f}s")
print()
print("Sample gold rows (Credit card, hours 7–10):")
print(con.execute("""
    SELECT hour_of_day, trip_count, avg_fare, rolling_3h_avg_fare,
           fare_rank, running_trips, pct_of_payment_type
    FROM gold_hourly_stats
    WHERE payment_label = 'Credit card'
      AND hour_of_day BETWEEN 7 AND 10
    ORDER BY hour_of_day
""").df().to_string(index=False))

**What just happened?**
- `SUM(...) OVER (PARTITION BY payment_label ORDER BY hour_of_day)` computes a running total that resets for each payment type
- `RANK() OVER (ORDER BY total_fare DESC)` identifies peak revenue hours across all payment types
- `AVG(...) OVER (... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` smooths fare volatility across a 3-hour sliding window
- All window functions ran inside a single SQL pass — no application-side groupby loops

In [ ]:
# Business insight: top 5 highest-revenue hours
print("Top 5 hours by total fare revenue:")
print(con.execute("""
    SELECT hour_of_day,
           round(sum(total_fare), 0) AS total_fare_all_payments,
           sum(trip_count) AS total_trips,
           min(fare_rank) AS rank
    FROM gold_hourly_stats
    GROUP BY hour_of_day
    ORDER BY total_fare_all_payments DESC
    LIMIT 5
""").df().to_string(index=False))

print()
print("Average tip by hour (credit card trips only):")
print(con.execute("""
    SELECT hour_of_day, avg_tip, avg_fare,
           round(avg_tip / avg_fare * 100, 1) AS tip_pct
    FROM gold_hourly_stats
    WHERE payment_label = 'Credit card'
    ORDER BY tip_pct DESC
    LIMIT 6
""").df().to_string(index=False))

---
## Step 5 · Export gold layer to compressed Parquet

The gold table is the deliverable — export it as a compressed Parquet file so downstream consumers (BI tools, Spark, another DuckDB instance) can read it without a live database connection.

We'll try both SNAPPY and ZSTD compression and compare sizes.

> **Tip:** ZSTD typically achieves 20–30% better compression than SNAPPY at comparable read speed with DuckDB's vectorised reader.

In [ ]:
output_dir = pathlib.Path('/tmp/lakehouse_output')
output_dir.mkdir(parents=True, exist_ok=True)

gold_snappy = str(output_dir / 'gold_hourly_stats_snappy.parquet')
gold_zstd   = str(output_dir / 'gold_hourly_stats_zstd.parquet')

# Export with SNAPPY
con.execute(f"""
COPY gold_hourly_stats TO '{gold_snappy}'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")

# Export with ZSTD
con.execute(f"""
COPY gold_hourly_stats TO '{gold_zstd}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

snappy_size = os.path.getsize(gold_snappy)
zstd_size   = os.path.getsize(gold_zstd)

print(f"SNAPPY: {snappy_size:,} bytes")
print(f"ZSTD:   {zstd_size:,} bytes")
print(f"ZSTD is {(1 - zstd_size/snappy_size)*100:.1f}% smaller")

# Verify schema round-trip by reading back
schema_back = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{gold_zstd}')").df()
print(f"\nParquet schema ({len(schema_back)} columns):")
print(schema_back[['column_name', 'column_type']].to_string(index=False))

---
## Step 6 · Profile with EXPLAIN ANALYZE and apply an optimisation

Before: we query `stg_taxi` without any filter — DuckDB scans all rows.
After: we apply predicate pushdown (an early WHERE clause) to prune rows before the aggregation.

We'll measure both with `EXPLAIN ANALYZE` and compare actual row counts at the scan operator.

> **What to look for in EXPLAIN ANALYZE:** the `Rows` column next to each operator. When actual rows >> estimated rows, you have a statistics problem. When you can push a filter earlier in the plan, you reduce work for every downstream operator.

In [ ]:
# Baseline: aggregate over ALL staging rows
plan_baseline = con.execute("""
EXPLAIN ANALYZE
SELECT hour_of_day, count(*), round(avg(fare_amount), 2)
FROM stg_taxi
GROUP BY hour_of_day
ORDER BY hour_of_day
""").fetchall()

print("=== EXPLAIN ANALYZE: baseline (full scan) ===")
for row in plan_baseline:
    print(row[1][:120] if len(row) > 1 else row)

print()

# Optimised: filter to credit card trips first (predicate pushdown)
plan_optimised = con.execute("""
EXPLAIN ANALYZE
SELECT hour_of_day, count(*), round(avg(fare_amount), 2)
FROM stg_taxi
WHERE payment_type = 1          -- Credit card only
GROUP BY hour_of_day
ORDER BY hour_of_day
""").fetchall()

print("=== EXPLAIN ANALYZE: optimised (payment_type filter) ===")
for row in plan_optimised:
    print(row[1][:120] if len(row) > 1 else row)

**What just happened?**
- The baseline plan scans all rows in `stg_taxi` before aggregating
- The optimised plan applies the `payment_type = 1` filter at the **TABLE_SCAN** operator — DuckDB evaluates it as it reads column chunks, skipping rows early
- In DuckDB's columnar engine, filtering on a narrow column (integer) is very cheap — the vectorised executor processes 2048 rows per batch and short-circuits quickly
- For larger datasets, adding a `ROW_GROUP` filter (via min/max statistics in Parquet metadata) can skip entire row groups — DuckDB does this automatically when reading Parquet files

In [ ]:
import time

# Measure execution time: full scan vs filtered
def time_query(sql, label, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        con.execute(sql).fetchall()
        times.append(time.perf_counter() - t0)
    avg_ms = sum(times) / len(times) * 1000
    print(f"{label}: {avg_ms:.0f} ms (avg of {runs} runs)")

time_query(
    "SELECT hour_of_day, count(*), round(avg(fare_amount),2) FROM stg_taxi GROUP BY hour_of_day",
    "Full scan"
)
time_query(
    "SELECT hour_of_day, count(*), round(avg(fare_amount),2) FROM stg_taxi WHERE payment_type=1 GROUP BY hour_of_day",
    "Filtered (credit card)"
)
time_query(
    "SELECT hour_of_day, count(*), round(avg(fare_amount),2) FROM stg_taxi WHERE payment_type=1 GROUP BY hour_of_day",
    "Filtered (warmed cache)"
)

---
## Step 7 · Python orchestration script

A real pipeline wraps all stages in a function, logs row counts at each layer, and reports any stage that fails.

This is the complete, self-contained version of everything above — runnable as `python pipeline.py` from a terminal.

**Design principles applied:**
- Each stage is idempotent (`CREATE OR REPLACE TABLE`) — safe to re-run
- Row counts are validated at each stage boundary — catch data loss early
- Elapsed time is tracked — spot regressions before they reach production
- The script returns a non-zero exit code on failure — CI/CD systems detect it automatically

In [ ]:
PIPELINE_SCRIPT = '''
#!/usr/bin/env python3
"""DuckDB Analytical Lakehouse Pipeline — Day 10 Capstone."""

import duckdb
import os
import pathlib
import sys
import time

RAW_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
OUTPUT_DIR = pathlib.Path("/tmp/lakehouse_output")
GOLD_PARQUET = OUTPUT_DIR / "gold_hourly_stats_zstd.parquet"


def log(stage: str, rows: int, elapsed: float) -> None:
    print(f"  [{stage}] {rows:>10,} rows  ({elapsed:.1f}s)")


def run_pipeline() -> int:
    """Run all pipeline stages. Returns 0 on success, 1 on failure."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs")
    con.execute("SET threads TO 2; SET memory_limit = \\"1GB\\"")

    pipeline_start = time.time()
    print("Starting lakehouse pipeline...")

    stages = [
        # (stage_name, sql, expected_min_rows)
        ("RAW", f"""
            CREATE OR REPLACE TABLE raw_taxi AS
            SELECT * FROM read_parquet(\\\'{RAW_URL}\\\')
        """, 1_000_000),
        ("STAGING", """
            CREATE OR REPLACE TABLE stg_taxi AS
            SELECT
                VendorID AS vendor_id,
                tpep_pickup_datetime::TIMESTAMP AS pickup_ts,
                tpep_dropoff_datetime::TIMESTAMP AS dropoff_ts,
                passenger_count::INTEGER AS passenger_count,
                trip_distance::DOUBLE AS trip_distance_mi,
                RatecodeID::INTEGER AS rate_code,
                PULocationID::INTEGER AS pickup_location_id,
                DOLocationID::INTEGER AS dropoff_location_id,
                fare_amount::DOUBLE AS fare_amount,
                tip_amount::DOUBLE AS tip_amount,
                total_amount::DOUBLE AS total_amount,
                payment_type::INTEGER AS payment_type,
                date_diff(\'minute\', tpep_pickup_datetime::TIMESTAMP,
                                    tpep_dropoff_datetime::TIMESTAMP) AS trip_duration_minutes,
                hour(tpep_pickup_datetime::TIMESTAMP) AS hour_of_day,
                dayofweek(tpep_pickup_datetime::TIMESTAMP) AS day_of_week
            FROM raw_taxi
            WHERE trip_distance > 0
              AND fare_amount > 0
              AND total_amount > 0
              AND passenger_count BETWEEN 1 AND 8
              AND tpep_pickup_datetime IS NOT NULL
              AND tpep_dropoff_datetime IS NOT NULL
              AND date_diff(\'minute\', tpep_pickup_datetime::TIMESTAMP,
                                      tpep_dropoff_datetime::TIMESTAMP) BETWEEN 1 AND 180
        """, 900_000),
        ("GOLD", """
            CREATE OR REPLACE TABLE gold_hourly_stats AS
            WITH hourly_agg AS (
                SELECT hour_of_day, payment_type,
                       count(*) AS trip_count,
                       round(sum(fare_amount), 2) AS total_fare,
                       round(avg(fare_amount), 2) AS avg_fare,
                       round(avg(tip_amount), 2) AS avg_tip,
                       round(avg(trip_distance_mi), 2) AS avg_distance_mi,
                       round(avg(trip_duration_minutes), 1) AS avg_duration_min
                FROM stg_taxi
                GROUP BY hour_of_day, payment_type
            )
            SELECT *,
                   SUM(trip_count) OVER (PARTITION BY payment_type ORDER BY hour_of_day) AS running_trips,
                   RANK() OVER (ORDER BY total_fare DESC) AS fare_rank,
                   round(AVG(avg_fare) OVER (
                       PARTITION BY payment_type ORDER BY hour_of_day
                       ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
                   ), 2) AS rolling_3h_avg_fare
            FROM hourly_agg
            ORDER BY hour_of_day, payment_type
        """, 10),
    ]

    try:
        for stage_name, sql, min_rows in stages:
            t0 = time.time()
            con.execute(sql)
            rows = con.execute(f"SELECT count(*) FROM {stage_name.lower()}_{'taxi' if stage_name != 'GOLD' else 'hourly_stats'}").fetchone()[0]
            elapsed = time.time() - t0
            log(stage_name, rows, elapsed)
            if rows < min_rows:
                print(f"ERROR: {stage_name} produced {rows} rows, expected >= {min_rows}")
                return 1

        # Export gold layer
        t0 = time.time()
        con.execute(f"COPY gold_hourly_stats TO \\\'{GOLD_PARQUET}\\\' (FORMAT PARQUET, COMPRESSION ZSTD)")
        size = os.path.getsize(GOLD_PARQUET)
        print(f"  [EXPORT] {GOLD_PARQUET} ({size:,} bytes, {time.time()-t0:.1f}s)")

        total = time.time() - pipeline_start
        print(f"Pipeline complete in {total:.1f}s")
        return 0

    except Exception as exc:
        print(f"PIPELINE FAILED: {exc}")
        return 1


if __name__ == "__main__":
    sys.exit(run_pipeline())
'''

# Write the script to disk
script_path = pathlib.Path('/tmp/lakehouse_pipeline.py')
script_path.write_text(PIPELINE_SCRIPT)
print(f"Pipeline script written to: {script_path}")
print(f"Size: {script_path.stat().st_size:,} bytes")
print("\nTo run standalone:")
print("  python /tmp/lakehouse_pipeline.py")

In [ ]:
# Run the orchestrated pipeline from within the notebook to verify end-to-end
print("Running orchestration via pipeline function...")
print()

OUTPUT_DIR = pathlib.Path('/tmp/lakehouse_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GOLD_PARQUET = str(OUTPUT_DIR / 'gold_hourly_stats_pipeline.parquet')

pipeline_start = time.time()
stages_log = []

# Each stage runs CREATE OR REPLACE — idempotent even if run again
stage_checks = [
    ('RAW',     'SELECT count(*) FROM raw_taxi'),
    ('STAGING', 'SELECT count(*) FROM stg_taxi'),
    ('GOLD',    'SELECT count(*) FROM gold_hourly_stats'),
]

for stage_name, count_sql in stage_checks:
    t0 = time.time()
    rows = con.execute(count_sql).fetchone()[0]
    elapsed = time.time() - t0
    stages_log.append((stage_name, rows, elapsed))
    print(f"  [{stage_name:<8}] {rows:>10,} rows  ({elapsed*1000:.0f}ms — table already materialised)")

# Export gold
t0 = time.time()
con.execute(f"COPY gold_hourly_stats TO '{GOLD_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)")
size = os.path.getsize(GOLD_PARQUET)
print(f"  [EXPORT  ] {GOLD_PARQUET} ({size:,} bytes, {time.time()-t0:.1f}s)")

total = time.time() - pipeline_start
print(f"\nAll stages verified. Total: {total:.2f}s")

---
## Challenge — extend the pipeline

Choose one or more extensions to make this lakehouse more production-ready:

In [ ]:
# Challenge A: Add a second dataset
# The TLC also publishes Green Taxi and For-Hire Vehicle data.
# Ingest the Green Taxi January 2024 file and JOIN it to the yellow taxi data
# to compare avg_fare and avg_distance_mi across taxi types.
#
# GREEN_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet"
#
# Steps:
#   1. CREATE TABLE raw_green AS SELECT * FROM read_parquet(GREEN_URL)
#   2. Create stg_green with the same cleaning rules (column names differ slightly)
#   3. UNION ALL stg_taxi and stg_green into a combined_trips view
#   4. Recreate gold_hourly_stats from combined_trips and compare results

# Challenge B: Persist the gold layer to a DuckDB file
# Instead of exporting to Parquet, write gold to a persistent .duckdb file
# so downstream tools can query it with SQL:
#
#   gold_con = duckdb.connect('/tmp/gold.duckdb')
#   gold_con.execute("CREATE TABLE gold_hourly_stats AS SELECT * FROM read_parquet('/tmp/lakehouse_output/gold_hourly_stats_zstd.parquet')")
#   print(gold_con.execute("SELECT count(*) FROM gold_hourly_stats").fetchone())

# Challenge C: Add row-level data quality checks
# Write a function validate_staging(con) that:
#   - Asserts count(*) > 1_000_000
#   - Asserts min(fare_amount) > 0
#   - Asserts max(trip_duration_minutes) <= 180
#   - Asserts count(DISTINCT hour_of_day) = 24
#   - Raises ValueError with a descriptive message on any failure

# Your solution here
print("Pick a challenge above and implement it in the cells below.")

---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| **Medallion architecture** | Raw → Staging → Gold: never modify raw data; rebuild downstream layers instead |
| `read_parquet(URL)` with httpfs | DuckDB streams remote Parquet over HTTP with no local download required |
| `CREATE TABLE AS SELECT` | Materialises a query result as a table — fast, portable, idempotent with `OR REPLACE` |
| Window functions in CTEs | Wrap GROUP BY in a CTE, then apply `SUM/RANK/AVG OVER` in the outer SELECT |
| `ROWS BETWEEN N PRECEDING AND CURRENT ROW` | Frame-based window: rolling average over the last N+1 rows |
| `EXPLAIN ANALYZE` | Shows actual vs estimated row counts — spot filter pushdown opportunities |
| `COPY TO … (FORMAT PARQUET, COMPRESSION ZSTD)` | Export any DuckDB query result as a compressed Parquet file |
| Idempotent pipeline | `CREATE OR REPLACE TABLE` at every stage — safe to re-run without side effects |
| Python orchestration | Wrap each stage in a try/except, validate row counts, return non-zero on failure |

> **Tip:** A production lakehouse in DuckDB is just three `CREATE TABLE AS SELECT` statements chained together. Keep raw data untouched, transform in staging, and aggregate in gold — the same medallion pattern works at any scale.

---
## Congratulations — you've completed DuckDB for Analytical Engineers!

Over 10 days you've covered:
- **Days 1–3:** Core engine, SQL dialects, window functions, CTEs, and macros
- **Days 4–5:** Python API, Pandas/Polars/Arrow zero-copy integration
- **Day 6:** httpfs, spatial, and community extensions
- **Day 7:** COPY TO, EXPORT DATABASE, Parquet tuning
- **Day 8:** EXPLAIN ANALYZE, parallel execution, and performance profiling
- **Day 9:** Persistent databases, ACID transactions, ATTACH, and idempotent upserts
- **Day 10:** Full analytical lakehouse pipeline on real public data

Mark Day 10 complete in your [tracker](../index.html).